# 05 - Backbone Fine-Tuning Comparison
Compare the same backbone with and without backbone training using the full `ArcFaceModel` end-to-end pipeline.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import torch
from dotenv import load_dotenv

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import src.data as data
import src.inference as inference
import src.wandb_utils as wandb_utils
from src.models import ArcFaceModel
from src.training import build_optimizer, fit
from src.utils import get_device, set_seed

load_dotenv(project_root / ".env")

RANDOM_SEED = 42
set_seed(RANDOM_SEED)

device = get_device()
device

device(type='cuda')

## Base Config

In [ ]:
config = {
    # Paths
    "data_dir": Path("../data"),
    "checkpoint_dir": Path("../checkpoints/e2_backbone_finetune"),
    "submission_dir": Path("../submissions/e2_backbone_finetune"),

    # Model
    "embedding_dim": 256,
    "hidden_dim": 512,

    # ArcFace
    "arcface_margin": 0.5,
    "arcface_scale": 64.0,
    "dropout": 0.3,

    # Training
    "batch_size": 32,
    "learning_rate": 1e-4,
    "head_learning_rate": 1e-4,
    "backbone_learning_rate": 1e-5,
    "weight_decay": 1e-4,
    "num_epochs": 2,
    "patience": 10,
    "val_split": 0.2,
    "num_workers": 2,

    # Reproducibility
    "seed": RANDOM_SEED,

    # Image normalization for train/val augmentation pipeline
    "mean": data.DEFAULT_MEAN,
    "std": data.DEFAULT_STD,
}

config["rerank"] = {
    "enabled": True,
    "k1": 20,
    "k2": 6,
    "lambda_value": 0.3,
}

config["checkpoint_dir"].mkdir(parents=True, exist_ok=True)
config["submission_dir"].mkdir(parents=True, exist_ok=True)
config

{'data_dir': PosixPath('../data'),
 'checkpoint_dir': PosixPath('../checkpoints/e2_backbone_finetune'),
 'submission_dir': PosixPath('../submissions/e2_backbone_finetune'),
 'embedding_dim': 256,
 'hidden_dim': 512,
 'arcface_margin': 0.5,
 'arcface_scale': 64.0,
 'dropout': 0.3,
 'batch_size': 32,
 'learning_rate': 0.0001,
 'head_learning_rate': 0.0001,
 'backbone_learning_rate': 1e-05,
 'weight_decay': 0.0001,
 'num_epochs': 20,
 'patience': 10,
 'val_split': 0.2,
 'num_workers': 2,
 'seed': 42,
 'mean': (0.481, 0.457, 0.408),
 'std': (0.268, 0.261, 0.275),
 'rerank': {'enabled': True, 'k1': 20, 'k2': 6, 'lambda_value': 0.3}}

## Load Data

In [3]:
def load_data(config, backbone):
    train_df = data.load_train_df(config["data_dir"])
    train_df, label_encoder = data.encode_labels(train_df)
    num_classes = len(label_encoder.classes_)

    train_data, val_data = data.train_val_split(
        train_df,
        val_split=config["val_split"],
        seed=config["seed"],
        stratify_col="ground_truth",
    )

    train_loader, val_loader = data.create_dataloaders(
        train_data,
        val_data,
        img_dir=config["data_dir"] / "train" / "train",
        input_size=config["input_size"],
        batch_size=config["batch_size"],
        num_workers=config["num_workers"],
        mean=config["mean"],
        std=config["std"],
        weighted_sampling=False,
    )

    pairs_df = data.load_test_pairs_df(config["data_dir"])
    unique_images = sorted(set(pairs_df["query_image"].unique()) | set(pairs_df["gallery_image"].unique()))
    test_df = pd.DataFrame({"filename": unique_images})

    test_loader = data.create_test_loader(
        backbone,
        test_df,
        img_dir=config["data_dir"] / "test" / "test",
        input_size=config["input_size"],
        batch_size=config["batch_size"],
        num_workers=config["num_workers"],
    )

    print(
        f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}"
    )

    return {
        "label_encoder": label_encoder,
        "num_classes": num_classes,
        "train_loader": train_loader,
        "val_loader": val_loader,
        "pairs_df": pairs_df,
        "test_loader": test_loader,
    }


## Run Experiment

In [ ]:
experiments = [
    {
        "name": "EVA02_Large_frozen",
        "backbone_name": "eva02_large_patch14_448.mim_m38m_ft_in22k_in1k",
        "input_size": 448,
        "freeze_backbone": True,
        "backbone_learning_rate": 0.0,
    },
    # {
    #     "name": "EVA02_Large_finetune",
    #     "backbone_name": "eva02_large_patch14_448.mim_m38m_ft_in22k_in1k",
    #     "input_size": 448,
    #     "freeze_backbone": False,
    #     "backbone_learning_rate": 1e-5,
    # },
]


def run_experiment(experiment, run_idx, total_runs):
    print("=" * 80)
    print(f"Run {run_idx}/{total_runs}: {experiment['name']}")

    config["backbone_name"] = experiment["backbone_name"]
    config["input_size"] = experiment["input_size"]
    config["freeze_backbone"] = experiment["freeze_backbone"]
    config["backbone_learning_rate"] = experiment["backbone_learning_rate"]

    train_df = data.load_train_df(config["data_dir"])
    train_df, label_encoder = data.encode_labels(train_df)
    num_classes = len(label_encoder.classes_)

    model = ArcFaceModel(
        backbone_name=config["backbone_name"],
        num_classes=num_classes,
        embedding_dim=config["embedding_dim"],
        hidden_dim=config["hidden_dim"],
        margin=config["arcface_margin"],
        scale=config["arcface_scale"],
        dropout=config["dropout"],
        pretrained=True,
        freeze_backbone=config["freeze_backbone"],
    ).to(device)

    data_bundle = load_data(config, model.backbone)
    label_encoder = data_bundle["label_encoder"]
    num_classes = data_bundle["num_classes"]
    train_loader = data_bundle["train_loader"]
    val_loader = data_bundle["val_loader"]
    pairs_df = data_bundle["pairs_df"]
    test_loader = data_bundle["test_loader"]

    if model.arcface.num_classes != num_classes:
        raise ValueError(f"Model num_classes={model.arcface.num_classes} but dataset has {num_classes}")

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    backbone_params = sum(p.numel() for p in model.backbone.parameters())

    wandb = wandb_utils.init_wandb(
        config,
        run_name=experiment["name"],
        param_count=trainable_params,
        param_count_backbone=backbone_params,
    )
    wandb.run.summary["total_param_count"] = total_params
    wandb.run.summary["trainable_param_count"] = trainable_params

    criterion = torch.nn.CrossEntropyLoss()
    optimizer = build_optimizer(model, config)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=5,
    )

    checkpoint_name = f"{experiment['name']}.pth"
    checkpoint_path = config["checkpoint_dir"] / checkpoint_name
    results = fit(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        config=config,
        device=device,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        label_encoder=label_encoder,
        wandb_run=wandb,
        checkpoint_name=checkpoint_name,
    )

    wandb.run.summary["best_val_mAP"] = results["best_map"]
    wandb.run.summary["best_val_mAP_rerank"] = results.get("best_map_rerank")
    wandb.run.summary["best_val_loss"] = results["best_val_loss"]
    wandb.run.summary["best_epoch"] = results["best_epoch"]
    wandb.run.summary["total_epochs"] = results["epochs_ran"]

    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    plain_submission_path = config["submission_dir"] / f"submission_{experiment['name']}.csv"
    rerank_submission_path = config["submission_dir"] / f"submission_{experiment['name']}_rerank.csv"

    inference.create_submission_model(
        model,
        device,
        pairs_df,
        test_loader,
        output_path=plain_submission_path,
        use_rerank=False,
    )

    inference.create_submission_model(
        model,
        device,
        pairs_df,
        test_loader,
        output_path=rerank_submission_path,
        use_rerank=config["rerank"]["enabled"],
        k1=config["rerank"]["k1"],
        k2=config["rerank"]["k2"],
        lambda_value=config["rerank"]["lambda_value"],
    )

    wandb_utils.log_checkpoint_artifact(
        wandb,
        checkpoint_path,
        artifact_name=experiment["name"],
        description="checkpoint",
    )

    wandb_utils.log_submission_artifact(
        wandb,
        plain_submission_path,
        artifact_name=f"{experiment['name']}_plain",
        description="Competition submission file",
    )

    wandb_utils.log_submission_artifact(
        wandb,
        rerank_submission_path,
        artifact_name=f"{experiment['name']}_rerank",
        description="Competition submission file with reranking",
    )

    wandb.finish()
    print(f"Finished run {run_idx}")


for run_idx, experiment in enumerate(experiments, start=1):
    run_experiment(experiment, run_idx, len(experiments))


Run 1/1: EVA02_Large_finetune


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /sc/home/maximilian.speer/.netrc


Train batches: 48 | Val batches: 12 | Test batches: 12


wandb: Currently logged in as: maximilian-speer (maximilian-speer-hasso-plattner-institut) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Starting training for 20 epochs...

Epoch 1/20


...

Embedding:   0%|          | 0/12 [00:00<?, ?it/s]

Embedding:   0%|          | 0/12 [00:00<?, ?it/s]

  Train Loss: 4.1288 | Train Acc: 67.7%
  Val Loss:   3.6785 | Val Acc:   83.9%
  Val mAP:    0.8375 | LR: 1.00e-05
  Val mAP RR: 0.8455 | k1=20 k2=6 lambda=0.3
  [New best model saved]

Epoch 5/20


Training:   0%|          | 0/48 [00:00<?, ?it/s]

Validation:   0%|          | 0/12 [00:00<?, ?it/s]